In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch

OUT = Path("paper_figures")
OUT.mkdir(exist_ok=True)

# ตั้งค่าเริ่มต้นให้เหมาะกับ Paper แบบ 2 คอลัมน์ (ตัวหนังสือใหญ่ เส้นหนาชัดเจน)
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,
    "legend.fontsize": 10.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.0,
})

METHOD_COLORS = {
    "Single Holdout": "#444444",
    "Expanding Window": "#0072B2",
    "Sliding Window": "#D55E00",
}

def save_figure(fig, filename):
    fig.savefig(
        OUT / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.05,
        facecolor="white",
    )
    plt.close(fig)


# ============================================================
# FIGURE 2: Actual data boundaries (ออกแบบใหม่สำหรับวางใน 1 คอลัมน์)
# ============================================================

train_ends = [
    "2013-12-31",
    "2015-12-31",
    "2017-12-29",
    "2019-12-31",
    "2021-12-31",
]

sliding_starts = [
    "2001-06-15",
    "2002-01-02",
    "2004-01-02",
    "2006-01-03",
    "2008-01-02",
]

validation = [
    ("2014-01-02", "2015-12-31"),
    ("2016-01-04", "2017-12-29"),
    ("2018-01-02", "2019-12-31"),
    ("2020-01-02", "2021-12-31"),
    ("2022-01-03", "2023-12-29"),
]

tests = [
    ("2016-01-04", "2017-12-29"),
    ("2018-01-02", "2019-12-31"),
    ("2020-01-02", "2021-12-31"),
    ("2022-01-03", "2023-12-29"),
    ("2024-01-02", "2026-07-28"),
]

# ใช้ชื่อย่อบนแกน Y เพื่อให้กราฟมีพื้นที่แสดงผลแนวนอนมากขึ้นในคอลัมน์แคบ
rows = [{
    "label": "Holdout",
    "full_label": "Single Holdout",
    "Train": ("2001-06-15", "2019-12-31"),
    "Validation": ("2020-01-02", "2023-12-29"),
    "Test": ("2024-01-02", "2026-07-28"),
}]

for i in range(5):
    rows.append({
        "label": f"Exp F{i + 1}",
        "full_label": f"Expanding F{i + 1}",
        "Train": ("2001-06-15", train_ends[i]),
        "Validation": validation[i],
        "Test": tests[i],
    })

for i in range(5):
    rows.append({
        "label": f"Slide F{i + 1}",
        "full_label": f"Sliding F{i + 1}",
        "Train": (sliding_starts[i], train_ends[i]),
        "Validation": validation[i],
        "Test": tests[i],
    })

split_colors = {
    "Train": "#0072B2",
    "Validation": "#E69F00",
    "Test": "#009E73",
}

# ปรับสัดส่วนเป็นแนวตั้ง (กว้าง 5.4 นิ้ว, สูง 6.2 นิ้ว)
fig, ax = plt.subplots(figsize=(5.4, 6.2))

ax.axvspan(
    pd.Timestamp("2024-01-02"),
    pd.Timestamp("2026-07-29"),
    color="#BBBBBB",
    alpha=0.25,
    zorder=0,
)

split_records = []

for y, row in enumerate(rows):
    for split in ["Train", "Validation", "Test"]:
        start, end = row[split]
        left = mdates.date2num(pd.Timestamp(start))
        right = mdates.date2num(pd.Timestamp(end) + pd.Timedelta(days=1))

        ax.barh(
            y,
            right - left,
            left=left,
            height=0.68,
            color=split_colors[split],
        )

        split_records.append({
            "experiment": row["full_label"],
            "split": split,
            "start": start,
            "end": end,
        })

ax.set_yticks(range(len(rows)))
ax.set_yticklabels([row["label"] for row in rows], fontweight="medium")
ax.invert_yaxis()

ax.axhline(0.5, color="0.6", linewidth=1.0, linestyle="--")
ax.axhline(5.5, color="0.6", linewidth=1.0, linestyle="--")

ax.xaxis_date()
ax.xaxis.set_major_locator(mdates.YearLocator(4))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.set_xlim(
    pd.Timestamp("2001-01-01"),
    pd.Timestamp("2027-01-01"),
)
ax.set_xlabel("Calendar year", fontweight="bold")
ax.grid(axis="x", alpha=0.25, linestyle=":")

legend_items = [
    Patch(color=split_colors["Train"], label="Train"),
    Patch(color=split_colors["Validation"], label="Validation"),
    Patch(color=split_colors["Test"], label="Test"),
    Patch(color="#BBBBBB", alpha=0.4, label="Common OOS"),
]
ax.legend(
    handles=legend_items,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.01),
    ncol=2,
    frameon=True,
)

fig.tight_layout()
save_figure(fig, "figure_2_temporal_splits")

pd.DataFrame(split_records).to_csv(
    OUT / "figure_2_source_dates.csv",
    index=False,
)


# ============================================================
# FIGURE 4: Reported fold metrics (แนวตั้ง + ตัวเลขกำกับบนแท่ง)
# ============================================================

metrics = pd.DataFrame({
    "fold": [1, 2, 3, 4, 5],
    "expanding_sharpe": [2.09, 1.01, 0.38, -0.09, 2.01],
    "sliding_sharpe": [2.09, 1.06, 0.58, 0.18, 2.44],
    "expanding_maxdd_pct": [-5.56, -10.80, -13.76, -31.60, -9.77],
    "sliding_maxdd_pct": [-5.56, -10.59, -13.67, -26.83, -9.11],
})

x = np.arange(len(metrics))
width = 0.38

fig, axes = plt.subplots(2, 1, figsize=(5.4, 6.4), sharex=True)

# --- Subplot 1: Sharpe Ratio ---
rects1 = axes[0].bar(
    x - width / 2,
    metrics["expanding_sharpe"],
    width,
    color=METHOD_COLORS["Expanding Window"],
    label="Expanding Window",
)
rects2 = axes[0].bar(
    x + width / 2,
    metrics["sliding_sharpe"],
    width,
    color=METHOD_COLORS["Sliding Window"],
    label="Sliding Window",
)
axes[0].axhline(0, color="black", linewidth=0.9)
axes[0].set_ylabel("Reported Sharpe", fontweight="bold")
axes[0].set_ylim(-0.5, 2.9)
axes[0].grid(axis="y", alpha=0.25)
axes[0].set_axisbelow(True)

# ใส่ตัวเลขกำกับบนแท่งกราฟ Sharpe
axes[0].bar_label(rects1, fmt="%.2f", padding=2, fontsize=8.5)
axes[0].bar_label(rects2, fmt="%.2f", padding=2, fontsize=8.5)

# --- Subplot 2: Maximum Drawdown ---
rects3 = axes[1].bar(
    x - width / 2,
    metrics["expanding_maxdd_pct"],
    width,
    color=METHOD_COLORS["Expanding Window"],
    label="Expanding Window",
)
rects4 = axes[1].bar(
    x + width / 2,
    metrics["sliding_maxdd_pct"],
    width,
    color=METHOD_COLORS["Sliding Window"],
    label="Sliding Window",
)
axes[1].axhline(0, color="black", linewidth=0.9)
axes[1].set_ylabel("Max Drawdown (%)", fontweight="bold")
axes[1].set_ylim(-37, 0)
axes[1].grid(axis="y", alpha=0.25)
axes[1].set_axisbelow(True)

# ใส่ตัวเลขกำกับใต้แท่งกราฟ MaxDD
axes[1].bar_label(rects3, fmt="%.1f", padding=2, fontsize=8.5)
axes[1].bar_label(rects4, fmt="%.1f", padding=2, fontsize=8.5)

axes[0].legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=2,
    frameon=True,
)

axes[1].set_xticks(x)
axes[1].set_xticklabels([
    "F1\n'16–'17",
    "F2\n'18–'19",
    "F3\n'20–'21",
    "F4\n'22–'23",
    "F5\n'24–'26*",
], fontweight="medium")

fig.tight_layout()
save_figure(fig, "figure_4_reported_fold_metrics")

metrics.to_csv(
    OUT / "figure_4_source_metrics.csv",
    index=False,
)


# ============================================================
# FIGURE 5: Forward account snapshots (แก้ปัญหาขอบขวาตกหล่น)
# ============================================================

snapshot_data = {
    "Single Holdout": {
        "timestamp": [
            "2026-10-05 15:36:35",
            "2026-10-06 16:32:01",
            "2026-10-07 16:15:12",
            "2026-10-08 14:13:02",
            "2026-10-09 18:26:10",
        ],
        "net_liq": [
            999992.29,
            1008249.25,
            1000567.13,
            1002132.95,
            1008544.74,
        ],
    },
    "Expanding Window": {
        "timestamp": [
            "2026-10-05 15:36:30",
            "2026-10-06 16:24:33",
            "2026-10-07 16:15:09",
            "2026-10-08 14:12:40",
            "2026-10-09 18:26:10",
        ],
        "net_liq": [
            999983.04,
            1008575.80,
            1000457.19,
            1001929.33,
            1008510.25,
        ],
    },
    "Sliding Window": {
        "timestamp": [
            "2026-10-05 15:36:24",
            "2026-10-06 19:59:50",
            "2026-10-07 16:15:17",
            "2026-10-08 14:12:46",
            "2026-10-09 18:25:51",
        ],
        "net_liq": [
            1000000.00,
            1007253.94,
            1000519.00,
            1001961.33,
            1008458.66,
        ],
    },
}

markers = {
    "Single Holdout": "o",
    "Expanding Window": "s",
    "Sliding Window": "^",
}

fig, ax = plt.subplots(figsize=(5.4, 4.5))
snapshot_records = []

for method, values in snapshot_data.items():
    timestamps = pd.to_datetime(values["timestamp"])
    net_liq = np.asarray(values["net_liq"], dtype=float)
    index_value = 100.0 * net_liq / net_liq[0]

    ax.plot(
        timestamps,
        index_value,
        marker=markers[method],
        linestyle="--",
        linewidth=1.6,
        markersize=7,
        color=METHOD_COLORS[method],
        label=method,
    )

    for timestamp, value, index_point in zip(
        values["timestamp"], net_liq, index_value
    ):
        snapshot_records.append({
            "method": method,
            "reported_timestamp": timestamp,
            "timezone": "unspecified_in_report",
            "net_liq": float(value),
            "first_snapshot_100": float(index_point),
        })

ax.axhline(100, color="0.5", linewidth=1.0, linestyle=":")
ax.set_ylabel("Account Index (First = 100)", fontweight="bold")
ax.set_xlabel("Reported Date (Oct 2026)", fontweight="bold")

# กำหนดขอบเขตแกน X เผื่อด้านซ้ายและขวา ไม่ให้จุดที่ 5 ตกขอบเหมือนในรูปเดิม
ax.set_xlim(
    pd.Timestamp("2026-10-05 06:00:00"),
    pd.Timestamp("2026-10-10 02:00:00"),
)
ax.set_ylim(99.95, 100.98)

ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.grid(alpha=0.25)

ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=2,
    frameon=True,
)

fig.tight_layout()
save_figure(fig, "figure_5_forward_snapshots")

pd.DataFrame(snapshot_records).to_csv(
    OUT / "figure_5_source_snapshots.csv",
    index=False,
)

print("Created figures and source tables in:", OUT.resolve())

Created figures and source tables in: D:\Trading Algorithm Project (Github)\Trading-Algorithm-Seminar-\Work15\paper_figures
